In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [8]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.86s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [ ]:
def forward_sample(prompt, top_k=4, top_p=0.9, n_steps=3):
    totalIterations = 0; 
    # Tokenize the input prompt
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    def sample_step(input_ids, depth):
        # Base case: If depth is zero, return the input_ids as a possible sequence
        if depth == 0:
            return [input_ids]

        # Run the model to get logits
        with torch.no_grad():
            outputs = model(input_ids=input_ids)
            logits = outputs.logits[:, -1, :]  # Take the logits of the last token

        # Calculate probabilities
        probs = torch.softmax(logits, dim=-1).squeeze()

        # Get top-k and top-p candidates
        top_k_probs, top_k_indices = torch.topk(probs, top_k)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=0)
        top_p_mask = cumulative_probs <= top_p
        top_p_indices = sorted_indices[top_p_mask]

        # Select the minimum of top-k or top-p candidates
        selected_indices = top_k_indices if len(top_k_indices) < len(top_p_indices) else top_p_indices
        sequences = []
        
        # Expand each selected token by recursively calling sample_step
        for i in range(len(selected_indices)):
            print(f"Iteration: {totalIterations}, Depth: {depth}, Step: {i}, Token: {tokenizer.decode(selected_indices[i].unsqueeze(0))}                 ", end="\r")
            next_token_id = selected_indices[i].unsqueeze(0).unsqueeze(0)
            next_input_ids = torch.cat([input_ids, next_token_id], dim=-1)
            sequences.extend(sample_step(next_input_ids, depth - 1))
            iterations += 1
        
        return sequences

    # Start sampling from the given prompt
    sampled_sequences = sample_step(input_ids, n_steps)

    # Decode all sampled sequences to strings
    decoded_sequences = [tokenizer.decode(seq[0], skip_special_tokens=True) for seq in sampled_sequences]
    
    return decoded_sequences

In [17]:
# Example usage
prompt = "Once upon a time"
sampled_texts = forward_sample(prompt, top_k=5, top_p=1, n_steps=5)
for text in sampled_texts:
    print(text)

Depth: 1, Step: 3, Token: .andernetlynn
Depth: 1, Step: 4, Token: 
Depth: 1, Step: 3, Token: .anderneton
Depth: 1, Step: 3, Token: .andernet
Depth: 1, Step: 4, Token: 
Depth: 1, Step: 4, Token: .whoendsn
Depth: 1, Step: 4, Token: .thatdslulion
Depth: 1, Step: 4, Token:  oferealntt
Depth: 1, Step: 4, Token:  therettent
Depth: 1, Step: 3, Token:  (heretyee
Depth: 1, Step: 4, Token: …
Depth: 1, Step: 4, Token:  oferesion
Depth: 1, Step: 3, Token: .ofacherknes
Depth: 1, Step: 2, Token: .artistsor
Depth: 1, Step: 2, Token: .withtssnship
Depth: 1, Step: 4, Token: ,
Depth: 1, Step: 2, Token: .ofargees
Depth: 1, Step: 3, Token: .toeamnent
Depth: 1, Step: 4, Token: .thather
Depth: 1, Step: 4, Token: .fordondnteyn
Depth: 1, Step: 4, Token: .aboutdhaphy
Depth: 2, Step: 2, Token: :calledeting
Depth: 1, Step: 0, Token: :following
Depth: 1, Step: 4, Token:  Ihereeordn
Once upon a time, there was a littlety
Once upon a time, there was a young
Once upon a time, there was a girl
Once upon a time, there

In [19]:
def get_coherence(text, window_size=4096):
    # Tokenize text
    tokens = tokenizer("\n" + text + "\n", return_tensors='pt', truncation=True, max_length=window_size)
    input_ids = tokens.input_ids
    
    # Split the text into smaller contexts (short and long contexts)
    short_context = input_ids[:, :window_size // 2]
    # long_context = input_ids
    
    # Predict on short and long contexts
    with torch.no_grad():
        outputs_short = model(short_context, labels=short_context)
        # outputs_long = model(long_context, labels=long_context)
    
    # Calculate accuracy and log-likelihood difference
    coherence_acc_short = (outputs_short.logits.argmax(-1) == short_context).float().mean().item()
    # coherence_acc_long = (outputs_long.logits.argmax(-1) == long_context).float().mean().item()
    
    #coherence_diff = (coherence_acc_long - coherence_acc_short) / coherence_acc_long
    return coherence_acc_short


In [20]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import numpy as np

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

model.eval()

def calculate_perplexity(text):
    inputs = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(inputs, labels=inputs)
        loss = outputs.loss
        perplexity = torch.exp(loss)
    return perplexity.item()

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [40]:
def calculate_log_likelihood_ratio(text):
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    # Calculate log-likelihood for each token chunk
    with torch.no_grad():
        log_likelihoods = []
        for i in range(1, len(input_ids[0]) - 1):
            partial_input = input_ids[:, :i + 1]
            outputs = model(partial_input, labels=partial_input)
            log_likelihoods.append(-outputs.loss.item())
        avg_likelihood = sum(log_likelihoods) / len(log_likelihoods)
    return avg_likelihood

likelihood_threshold = -0.3

In [41]:
data = [sampled_texts, [get_coherence(text) for text in sampled_texts], [calculate_perplexity(text) for text in sampled_texts], [calculate_log_likelihood_ratio(text) for text in sampled_texts]]

In [38]:
len(data[0])

3125